<!-- <center> -->
<h1> Using MOMENT for Classification </h1>
<!-- </center> -->
<hr>

## Contents
### 1. A Quick Introduction to Classification
### 2. Loading MOMENT
### 3. Inputs and Outputs
### 4. Two Approaches for Time Series Classification
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.1 Classification Dataset
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.2 Unsupervised Representation Learning using MOMENT
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.3 Learning a Statistical ML Classifier on MOMENT Embeddings
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.4 Visualize the Embeddings
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.5 Results Interpretation
#### &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 4.6 Fully Supervised Learning using Classification Head

## 1. A Quick Introduction to Classification

Classification is a popular time series modeling task which involves assigning a categorical label to a time series sub-sequence. For example, in the context of [electrocardiogram (ECG) recordings](https://en.wikipedia.org/wiki/Electrocardiography), classification may entail distinguishing between normal and anomalous heartbeats. In this tutorial, we will explore two ways to use MOMENT to solve any time series classification problem. Mathematically, the time series classification problem is defined as follow:

**Problem**: Given a time-series $T = [x_1, ..., x_L], \ x_i \in \mathbb{R}^{C}$ of length $L$ with $C$ channels (sensors or variables) with $n$ attributes, the classification problem is to predict a class label $m \in \{0, \dots, M\}$ for each time series.

## 2. Loading MOMENT

We will first install the MOMENT package, load some essential packages and the pre-trained model. 

MOMENT can be loaded in 4 modes: (1) `reconstruction`, (2) `embedding`, (3) `forecasting`, and (4) `classification`.

In the `reconstruction` mode, MOMENT reconstructs input time series, potentially containing missing values. We can solve imputation and anomaly detection problems in this mode. This mode is suitable for solving imputation and anomaly detection tasks. During pre-training, MOMENT is trained to predict the missing values within uniformly randomly masked patches (disjoint sub-sequences) of the input time series, leveraging information from observed data in other patches. As a result, MOMENT comes equipped with a pre-trained reconstruction head, enabling it to address imputation and anomaly detection challenges in a zero-shot manner! Check out the `anomaly_detection.ipynb` and `imputation.ipynb` notebooks for more details!

In the `embedding` model, MOMENT learns a $d$-dimensional embedding (e.g., $d=1024$ for `MOMENT-1-large`) for each input time series. These embeddings can be used for clustering and classification. MOMENT can learn embeddings in a zero-shot setting! Check out `representation_learning.ipynb` notebook for more details! 

The `forecasting` and `classification` modes are used for forecasting and classification tasks, respectively. In these modes, MOMENT learns representations which are subsequently mapped to the forecast horizon or the number of classes, using linear forecasting and classification heads. Both the forecasting and classification head are randomly initialized, and therefore must be fine-tuned before use. Check out the `forecasting.ipynb` notebook for more details!

In [ ]:
# !pip install numpy pandas scikit-learn matplotlib tqdm
# !pip install git+https://github.com/moment-timeseries-foundation-model/moment.git

In [ ]:
from momentfm import MOMENTPipeline

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={
        "task_name": "classification",
        "n_channels": 1,
        "num_class": 5,
    },  # We are loading the model in `classification` mode
    # local_files_only=True,  # Whether or not to only look at local files (i.e., do not try to download the model).
)

In [ ]:
model.init()
print(model)

## 3. Inputs and Outputs
Let's begin by performing a forward pass through MOMENT and examining its outputs!

MOMENT takes 3 inputs: 
1. An input time series of length $T=512$ timesteps and $C$ channels, and 
2. Two optional masks, both of length $T=512$. 
    - The input mask is utilized to regulate the time steps or patches that the model should attend to. For instance, in the case of shorter time series, you may opt not to attend to padding. To implement this, you can provide an input mask with zeros in the padded locations.  
    - The second mask, referred to simply as mask, denotes masked or unobserved values. We employ mask tokens to replace all patches containing any masked time step (for further details, refer to Section 3.2 in our [paper](https://arxiv.org/abs/2402.03885)). MOMENT can attend to these mask tokens during reconstruction.
    - By default, all time steps are observed and attended to.

MOMENT returns a `TimeseriesOutputs` object. Since this is a classification task, it returns both `logits` and `embeddings` of the input. 

In [ ]:
FORMAT = "npy"
from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
import numpy as np


def load_data(file_path):
    """Load and prepare the dataset"""
    if FORMAT == "ucr":
        ucr_uea_loader = UCR_UEA_datasets()
        X_train, y_train, X_test, y_test = ucr_uea_loader.load_dataset(file_path)

        # Reshape to put channel as second dimension
        if len(X_train.shape) == 3:
            # If already 3D, ensure channel is in middle
            X_train = np.transpose(X_train, (0, 2, 1))
            X_test = np.transpose(X_test, (0, 2, 1))
        else:
            # If 2D, add channel dimension in middle
            X_train = X_train.reshape(X_train.shape[0], 1, -1)
            X_test = X_test.reshape(X_test.shape[0], 1, -1)

        label_encoder = LabelEncoder()
        y_train = label_encoder.fit_transform(y_train)
        y_test = label_encoder.transform(y_test)

    elif FORMAT == "npy":
        data = np.load(file_path, allow_pickle=True).item()
        X_train = data["train"]["X"]
        X_test = data["test"]["X"]
        y_train = np.array([int(x) for x in data["train"]["y"]])
        y_test = np.array([int(x) for x in data["test"]["y"]])
    else:
        raise ValueError(f"Unsupported format: {FORMAT}")

    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = load_data("MP_centred.npy")

In [ ]:
from pprint import pprint
import torch

# takes in tensor of shape [batchsize, n_channels, context_length]
x = torch.randn(16, 1, 512)
output = model(x_enc=x)
pprint(output)

In [ ]:
# backward
# [batch_size, num_classes]
logits = output.logits

# [batch_size, ]
predicted_labels = logits.argmax(dim=1)
predicted_labels

**Note**: The classification head is randomly initialized, so these predictions are random. We must train the classification head to get reasonable results. Below we show a quick example of how we can fine-tune the classification head.

## 4. Two Approaches for Time Series Classification

We can use MOMENT to solve the classification problem in two ways: (1) unsupervised representation learning, and (2) fully supervised learning using classification head. 

### Unsupervised Representation Learning
In this setting, we use MOMENT to embed time series data (see `representation_learning.ipynb`). Next, we train a Support Vector Machine (SVM) classifier using these embeddings as features and labels. This setting is common in field of unsupervised representation learning, where the goal is to learn meaningful time series representations without any labeled data (see [TS2Vec](https://arxiv.org/pdf/2106.10466) for a recent example). The quality of these representations are evaluated based on the performance of the downstream classifier (in this case, SVM). This is also the setting that we consider in our [paper](https://arxiv.org/abs/2402.03885). 

### Fully Supervised Learning using Classification Head
In this approach, we replace MOMENT's reconstruction head with a classification head, which maps the patch-level represenations to the number of classes present in a dataset. This classification head is randomly initialized and can be trained using cross-entropy loss and labeled data. We will not explore this approach in detail in this tutorial.

### 4.1 Classification Dataset

For these experiments, we will use the ECG5000 dataset from the [UCR Classification Archive](https://www.timeseriesclassification.com/description.php?Dataset=ECG5000). The original dataset is a 20-hour long ECG record (chf07) from the BIDMC Congestive Heart Failure Database (CHFDB) available on [PhysioNet](https://physionet.org/about/database/). This data was pre-processed in two steps: (1) extract each heartbeat, (2) make each heartbeat equal length using interpolation. Following these steps, a subset of 5,000 heartbeats were randomly chosen and automatically annotated into 5 classes including normal and abnormal heartbeats.

We'll start by reading and pre-processing this dataset using the `ClassificationDataset` class. 

In [ ]:
from torch.utils.data import DataLoader
from momentfm.data.classification_dataset import ClassificationDataset

train_dataset = ClassificationDataset(data_split="train")
test_dataset = ClassificationDataset(data_split="test")

Now let's visualize the time series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

idx = np.random.randint(0, len(train_dataset))
heartbeat_start = np.argmax(train_dataset[idx][1])
heartbeat = train_dataset[idx][0].squeeze()[heartbeat_start:]
label = train_dataset[idx][2]
plt.plot(heartbeat, c="darkblue")
plt.title(f"idx={idx} | label={label}")
plt.show()

### 4.2 Unsupervised Representation Learning using MOMENT

In this setting, we use MOMENT to embed all training and testing time series. Then we train a statistical classifier (e.g. support vector machine) using the embeddings of the training time series as features and training labels. We will show that MOMENT can learn meaningful representations in a zero-shot setting, which can be used to train powerful statistical classifiers. 

Let's embed the train and test datasets! We'll proceed as follows: 
First, we will write a simple function `get_embedding` which will iterate over the training and testing datasets, and embed each time series. Then we will use the `fit_svm` function to fit a support vector machine (SVM) model using these embeddings as features and training labels. 

In [ ]:
train_dataloader = DataLoader(
    train_dataset, batch_size=64, shuffle=False, drop_last=False
)
test_dataloader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, drop_last=False
)

In [ ]:
from tqdm import tqdm


def get_embedding(model, dataloader):
    embeddings, labels = [], []
    with torch.no_grad():
        for batch_x, batch_masks, batch_labels in tqdm(
            dataloader, total=len(dataloader)
        ):
            batch_x = batch_x.to("cuda").float()
            batch_masks = batch_masks.to("cuda")

            output = model(
                x_enc=batch_x, input_mask=batch_masks
            )  # [batch_size x d_model (=1024)]
            embedding = output.embeddings
            embeddings.append(embedding.detach().cpu().numpy())
            labels.append(batch_labels)

    embeddings, labels = np.concatenate(embeddings), np.concatenate(labels)
    return embeddings, labels

For unsupervised representation learning, MOMENT can be initialized in both `embedding` and `classification` mode.

```python

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={'task_name': 'embedding'}, # We are loading the model in `embedding` mode
)
model.init()

```

In [ ]:
model.to("cuda").float()

train_embeddings, train_labels = get_embedding(model, train_dataloader)
test_embeddings, test_labels = get_embedding(model, test_dataloader)

print(train_embeddings.shape, train_labels.shape)
print(test_embeddings.shape, test_labels.shape)

### 4.3 Learning a Statistical ML Classifier on MOMENT Embeddings

In [ ]:
from momentfm.models.statistical_classifiers import fit_svm

clf = fit_svm(features=train_embeddings, y=train_labels)

y_pred_train = clf.predict(train_embeddings)
y_pred_test = clf.predict(test_embeddings)
train_accuracy = clf.score(train_embeddings, train_labels)
test_accuracy = clf.score(test_embeddings, test_labels)

print(f"Train accuracy: {train_accuracy:.2f}")
print(f"Test accuracy: {test_accuracy:.2f}")

### 4.4 Visualize the Embeddings

Next, let's visualize the embeddings that MOMENT is learning using Principal Component Analysis (PCA)

In [ ]:
from sklearn.decomposition import PCA

test_embeddings_manifold = PCA(n_components=2).fit_transform(test_embeddings)

plt.title(f"ECG5000 Test Embeddings", fontsize=20)
plt.scatter(
    test_embeddings_manifold[:, 0],
    test_embeddings_manifold[:, 1],
    c=test_labels.squeeze(),
)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

### 4.5 Results Interpretation 
#### MOMENT Learns Meaningful Representations without Dataset-specific Fine-tuning

Here we can see that the first two principal components of the representations that MOMENT is learning can distinguish normal and abnormal heartbeats denoted by different colors. This can explain MOMENT + SVM's promising classification accuracy. Note that these represenations are learnt zero-shot. Even without dataset-specific fine-tuning, MOMENT learns distinct representations of different classes of heartbeats.

### 4.6 Fully Supervised Learning using Classification Head


We can also perform classification by attaching a randomly-initialized classification head to MOMENT and fine-tuning it. Here's an example code to fine-tune MOMENT with a classification head:

```python

# Remember MOMENT must be initialized in `classification` mode
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': 1,
        'num_class': 5
    },
)

# Define a data loader 
train_dataloader = DataLoader(dataset, batch_size=64, shuffle=True) 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for data, labels in train_dataloader:
    # forward [batch_size, n_channels, forecast_horizon]
    output = model(x_enc=data)

    # backward
    loss = criterion(output.logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"loss: {loss.item():.3f}")
```